# AML-FT Multi-Agent Pipeline

1. **Agent 1**: Ingestão e qualidade dos dados
2. **Agent 2**: Feature Engineering  
3. **Agent 3**: ML Scoring (Fila priorizada)
4. **Agent 4**: SAR Report 

### **Bibliotecas e configurações**

In [1]:
import os, json, glob, pickle
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

from dotenv import load_dotenv
load_dotenv() 

from openai import OpenAI

In [2]:
RAW_DIR = "raw"
TRUSTED_DIR = "trusted"
FEATURES_DIR = "features"
ARTIFACTS_DIR = "artifacts"
MODELS_DIR = "models"

os.makedirs(TRUSTED_DIR, exist_ok=True)
os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

client = OpenAI() # Usa OPENAI_API_KEY automaticamente

In [3]:
def utc_now_iso() -> str:
    """Retorna a data e hora atual em UTC no formato ISO 8601."""
    return datetime.now(timezone.utc).isoformat()

def save_json(path: str, obj: dict) -> None:
    """Salva um dicionário em arquivo JSON com encoding UTF-8 e indentação."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_raw_tables(raw_dir: str) -> dict[str, pd.DataFrame]:
    """Carrega arquivos CSV e Parquet de um diretório e retorna como dicionário de DataFrames."""
    tables = {}
    for fp in glob.glob(os.path.join(raw_dir, "*")):
        name = os.path.splitext(os.path.basename(fp))[0]
        if fp.endswith(".csv"):
            tables[name] = pd.read_csv(fp)
        elif fp.endswith(".parquet"):
            tables[name] = pd.read_parquet(fp)
    if not tables:
        raise FileNotFoundError(f"No CSV/Parquet files found in {raw_dir}")
    return tables

def basic_profile(df: pd.DataFrame, key_candidates: list[str] | None = None) -> dict:
    """Gera um perfil básico do DataFrame com estatísticas de colunas, tipos e taxa de nulidade."""
    prof = {
        "rows": int(df.shape[0]),
        "cols": int(df.shape[1]),
        "columns": [{"name": c, "dtype": str(df[c].dtype), "null_rate": float(df[c].isna().mean())} for c in df.columns],
    }
    if key_candidates:
        dups = {}
        for k in key_candidates:
            if k in df.columns:
                dups[k] = float(df[k].duplicated().mean())
        prof["dup_rate"] = dups
    return prof

def llm_json(model: str, system: str, user: str, schema_name: str, schema: dict) -> dict:
    """Chama a API OpenAI Chat Completions com saída estruturada em JSON Schema e retorna o objeto parseado."""
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": schema_name,
                "strict": True,
                "schema": schema,
            },
        },
    )
    text = resp.choices[0].message.content
    return json.loads(text)

### **Agent 1: Ingestão e qualidade dos dados**
Define quality gates, plano de padronização e camada de dados confiável.

- Entrada:
     - arquivos CSV/Parquet em raw/
     - profiles estatísticos (gerados pelo Python)
- Saída lógica (JSON):
     - ingestion_result (schema v1: status, quality_score, ready_for_features,
       raw_inventory, trusted_outputs, tables_summary, warnings, etc.)
- Saída física:
     - trusted/*.parquet  (camada tratada)
     - artifacts/agent1_plan.json  (o JSON do Agent 1)
- Evento emitido:
     - ingestion.completed(status, ready_for_features, batch_id)

In [4]:
AGENT1_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "schema_version", "agent", "status", "quality_score", "ready_for_features",
        "raw_inventory", "standardizations_applied",
        "blocking_issues", "warnings", "assumptions", "provenance",
        "quality_gates", "cleaning_plan", "notes"
    ],
    "properties": {
        "schema_version": {"type": "string"},
        "agent": {"type": "string"},
        "status": {"type": "string", "enum": ["ok", "warn", "fail"]},
        "quality_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "ready_for_features": {"type": "boolean"},
        "raw_inventory": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["file", "type_guess", "notes"],
                "properties": {
                    "file": {"type": "string"},
                    "type_guess": {"type": "string", "enum": ["transactions", "kyc", "merchants", "signals", "other"]},
                    "notes": {"type": "string"},
                },
            },
        },
        "tables_summary": {
            "type": "object",
            "additionalProperties": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "row_count", "primary_key_candidates", "critical_columns",
                    "invalid_value_examples", "referential_integrity"
                ],
                "properties": {
                    "row_count": {"type": "integer"},
                    "primary_key_candidates": {"type": "array", "items": {"type": "string"}},
                    "critical_columns": {"type": "array", "items": {"type": "string"}},
                    "invalid_value_examples": {"type": "array", "items": {"type": "string"}},
                    "referential_integrity": {"type": "array", "items": {"type": "string"}},
                },
            },
        },
        "standardizations_applied": {"type": "array", "items": {"type": "string"}},
        "blocking_issues": {"type": "array", "items": {"type": "string"}},
        "warnings": {"type": "array", "items": {"type": "string"}},
        "assumptions": {"type": "array", "items": {"type": "string"}},
        "provenance": {
            "type": "object",
            "additionalProperties": False,
            "required": ["input_folder", "output_folder", "processed_at"],
            "properties": {
                "input_folder": {"type": "string"},
                "output_folder": {"type": "string"},
                "processed_at": {"type": "string"},
            },
        },
        "quality_gates": {
            "type": "object",
            "additionalProperties": False,
            "required": ["block_if", "warn_if"],
            "properties": {
                "block_if": {"type": "array", "items": {"type": "string"}},
                "warn_if": {"type": "array", "items": {"type": "string"}},
            },
        },
        "cleaning_plan": {
            "type": "object",
            "additionalProperties": False,
            "required": ["standardize_columns", "parse_timestamps", "drop_exact_duplicates", "mark_invalid_records"],
            "properties": {
                "standardize_columns": {"type": "boolean"},
                "parse_timestamps": {"type": "array", "items": {"type": "string"}},
                "drop_exact_duplicates": {"type": "array", "items": {"type": "string"}},
                "mark_invalid_records": {"type": "array", "items": {"type": "string"}},
            },
        },
        "trusted_tables": {
            "type": "object",
            "additionalProperties": {"type": "string"}
        },
        "notes": {"type": "array", "items": {"type": "string"}},
    },
}

AGENT1_SYSTEM = (
    "Você é o Agente 1 (Data Ingestion & Quality) AML-FT. "
    "Existe uma pasta 'raw' com arquivos de dados. Você deve definir como tratar "
    "(limpar/padronizar/validar) os dados brutos e produzir a camada 'trusted'. "
    "Regras: responda EXCLUSIVAMENTE em JSON válido conforme o schema, sem markdown. "
    "Não invente colunas/dados. Seja objetivo e auditável."
)


In [5]:
def agent1_ingest_and_quality(raw_tables: dict[str, pd.DataFrame], model: str = "gpt-4.1-mini") -> dict:
    """Gera um plano de qualidade e limpeza para as tabelas raw usando um LLM.

    Args:
        raw_tables: dicionário {nome: DataFrame} com as tabelas brutas carregadas.
        model: identificador do modelo LLM a ser usado.

    Returns:
        dict: plano conforme `AGENT1_SCHEMA` (contendo `status`, `quality_score`, `ready_for_features`,
        `raw_inventory`, `standardizations_applied`, `cleaning_plan`, `quality_gates`, etc.).

    Efeitos colaterais:
        - Persiste o plano em `artifacts/agent1_plan.json` via `save_json`.
        - Não modifica as tabelas raw.
    """
    # Build a lightweight profile for the LLM (no raw rows, só estatísticas)
    profiles = {}
    for name, df in raw_tables.items():
        key_candidates = ["transaction_id", "customer_id", "sender_id", "merchant_id"]
        profiles[name] = basic_profile(df, key_candidates=key_candidates)

    user = f"""
    Contexto:
    Existe uma pasta "raw" com arquivos de dados do case AML-FT. Você deve TRATAR (limpar/padronizar/validar) os dados brutos e produzir a camada "trusted" para o próximo agente.

    Você recebeu perfis (estatísticas) das tabelas raw e uma lista de arquivos encontrados.

    Tarefas obrigatórias:
    1) Inventariar raw:
    - Liste arquivos encontrados e infira o tipo (transactions, kyc, merchants, signals, outros).

    2) Padronizar:
    - Normalizar nomes de colunas (snake_case).
    - Padronizar tipos: IDs como string, timestamps ISO-8601, valores monetários numéricos.
    - Normalizar categorias (rail/channel/status) quando existirem.

    3) Validar qualidade:
    - Null rate em colunas críticas por tabela.
    - Duplicatas (ex.: transaction_id).
    - Valores inválidos (amount <= 0, timestamps inválidos).
    - Integridade referencial: transações referenciando customer_id/merchant_id inexistentes (se aplicável).

    4) Corrigir de forma segura (sem inventar dados):
    - Remover duplicatas exatas quando apropriado (registrar).
    - Converter tipos e datas; manter registro de linhas inválidas em invalid_record=true se necessário.

    5) Quality Gate raw→trusted:
    - status: ok|warn|fail
    - quality_score: 0–100
    - ready_for_features: boolean
    - blocking_issues e warnings

    6) Output:
    Retorne EXATAMENTE o JSON do schema (preencha todos os campos; use listas vazias quando não aplicável).

    Dados recebidos:
    - Pasta raw: "{RAW_DIR}"
    - Pasta trusted: "{TRUSTED_DIR}"
    - Perfis:
    {json.dumps(profiles, ensure_ascii=False)}

    - No campo provenance.processed_at use: "{utc_now_iso()}"
"""
    plan = llm_json(model, AGENT1_SYSTEM, user, "agent1_data_ingestion_quality", AGENT1_SCHEMA)
    save_json(os.path.join(ARTIFACTS_DIR, "agent1_plan.json"), plan)
    return plan


def apply_agent1_plan(raw_tables: dict[str, pd.DataFrame], plan: dict) -> dict[str, pd.DataFrame]:
    """Aplica o `cleaning_plan` gerado pelo Agent1 nas tabelas brutas e grava a camada `trusted`.

    Args:
        raw_tables: dicionário {nome: DataFrame} com as tabelas brutas.
        plan: dicionário conforme `AGENT1_SCHEMA` que contém instruções de limpeza/validação.

    Returns:
        dict: dicionário {nome: DataFrame} com as tabelas confiáveis (trusted), após as transformações.

    Comportamento principal:
        - Se `standardize_columns` for True, converte nomes de colunas para snake_case.
        - Tenta parsear colunas listadas em `parse_timestamps` para datetime UTC.
        - Remove duplicatas exatas para tabelas listadas em `drop_exact_duplicates`.
        - Marca registros inválidos conforme `mark_invalid_records` (ex.: `amount_brl <= 0`).
        - Persiste cada tabela em `{TRUSTED_DIR}/{table}.parquet`.

    Observações:
        - Implementação atual faz checagens simples (padrões "naive"); regras complexas devem ser expandas.
        - Espera que `plan["cleaning_plan"]` contenha as chaves requeridas pelo schema.
    """
    trusted = {}

    def to_snake(s: str) -> str:
        return (
            s.strip()
             .replace(" ", "_")
             .replace("-", "_")
             .replace(".", "_")
             .lower()
        )

    ts_cols = set(plan["cleaning_plan"]["parse_timestamps"])
    invalid_checks = plan["cleaning_plan"]["mark_invalid_records"]

    for name, df in raw_tables.items():
        df2 = df.copy()

        if plan["cleaning_plan"]["standardize_columns"]:
            df2.columns = [to_snake(c) for c in df2.columns]

        # parse timestamps if present
        for c in ts_cols:
            c2 = to_snake(c)
            if c2 in df2.columns:
                df2[c2] = pd.to_datetime(df2[c2], errors="coerce", utc=True)

        # drop exact duplicates for selected tables
        if name in plan["cleaning_plan"]["drop_exact_duplicates"]:
            df2 = df2.drop_duplicates()

        # invalid_record flag (very simple examples; you can expand)
        df2["invalid_record"] = False
        for rule in invalid_checks:
            # naive rule patterns; keep it simple and explicit
            if "amount_brl" in rule and " <= 0" in rule and "amount_brl" in df2.columns:
                df2.loc[df2["amount_brl"].fillna(0) <= 0, "invalid_record"] = True

        trusted[name] = df2

        out_path = os.path.join(TRUSTED_DIR, f"{name}.parquet")
        df2.to_parquet(out_path, index=False)

    return trusted



Fluxo de dados:
raw_tables (dicts de DataFrames brutos) → agent1_ingest_and_quality → plano JSON com instruções → apply_agent1_plan → trusted_tables (dicts de DataFrames limpos e validados) salvo em trusted/.

Responsabilidades separadas:
agent1_ingest_and_quality é responder (LLM decide o quê fazer com base em dados), apply_agent1_plan é executar (código determinístico aplica as decisões). Esta separação permite auditoria: o plano fica registrado em artifacts/agent1_plan.json para posterior análise de decisões tomadas.


### **Agent 2: Feature Engineering**

Gera especificações de features de AML-FT e calcula features a partir de dados confiáveis.

- Gate:
     - só prossegue se ingestion_result.ready_for_features == true
- Entrada:
     - ingestion_result (JSON do Agent 1)
     - trusted/transactions.parquet (+ kyc/merchants/signals se existirem)
- Saída lógica (JSON):
     - feature_engineering_result (schema v1: status, quality_score, ready_for_ml,
       feature_pack, feature_coverage, warnings, etc.)
- Saída física:
     - features/feature_table.parquet (dataset com colunas f_* etc.)
     - artifacts/agent2_feature_spec.json (ou agent2_result.json, se você salvar)
- Evento emitido:
     - features.completed(status, ready_for_ml, batch_id)

In [6]:
AGENT2_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "schema_version", "agent", "status", "quality_score", "ready_for_ml",
        "inputs", "enrichments_applied", "feature_pack",
        "feature_dataset_ref", "blocking_issues", "warnings", "assumptions", "provenance"
    ],
    "properties": {
        "schema_version": {"type": "string", "enum": ["v1"]},
        "agent": {"type": "string", "enum": ["feature_engineering"]},
        "status": {"type": "string", "enum": ["ok", "warn", "fail"]},
        "quality_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "ready_for_ml": {"type": "boolean"},
        "inputs": {
            "type": "object",
            "additionalProperties": False,
            "required": ["batch_id"],
            "properties": {
                "batch_id": {"type": "string"},
            },
        },
        "enrichments_applied": {"type": "array", "items": {"type": "string"}},
        "feature_pack": {
            "type": "object",
            "additionalProperties": False,
            "required": ["transaction_features", "entity_features"],
            "properties": {
                "transaction_features": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["name", "dtype", "definition", "source_cols"],
                        "properties": {
                            "name": {"type": "string"},
                            "dtype": {"type": "string"},
                            "definition": {"type": "string"},
                            "source_cols": {"type": "array", "items": {"type": "string"}},
                        },
                    },
                },
                "entity_features": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["customer", "merchant", "device", "ip"],
                    "properties": {
                        "customer": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "required": ["name", "dtype", "definition", "source_cols"],
                                "properties": {
                                    "name": {"type": "string"},
                                    "dtype": {"type": "string"},
                                    "definition": {"type": "string"},
                                    "source_cols": {"type": "array", "items": {"type": "string"}},
                                },
                            },
                        },
                        "merchant": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "required": ["name", "dtype", "definition", "source_cols"],
                                "properties": {
                                    "name": {"type": "string"},
                                    "dtype": {"type": "string"},
                                    "definition": {"type": "string"},
                                    "source_cols": {"type": "array", "items": {"type": "string"}},
                                },
                            },
                        },
                        "device": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "required": ["name", "dtype", "definition", "source_cols"],
                                "properties": {
                                    "name": {"type": "string"},
                                    "dtype": {"type": "string"},
                                    "definition": {"type": "string"},
                                    "source_cols": {"type": "array", "items": {"type": "string"}},
                                },
                            },
                        },
                        "ip": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "required": ["name", "dtype", "definition", "source_cols"],
                                "properties": {
                                    "name": {"type": "string"},
                                    "dtype": {"type": "string"},
                                    "definition": {"type": "string"},
                                    "source_cols": {"type": "array", "items": {"type": "string"}},
                                },
                            },
                        },
                    },
                },
            },
        },
        "feature_coverage": {"type": "object", "additionalProperties": {"type": "number"}},
        "feature_dataset_ref": {"type": "string"},
        "blocking_issues": {"type": "array", "items": {"type": "string"}},
        "warnings": {"type": "array", "items": {"type": "string"}},
        "assumptions": {"type": "array", "items": {"type": "string"}},
        "provenance": {
            "type": "object",
            "additionalProperties": False,
            "required": ["input_folder", "output_folder", "processed_at"],
            "properties": {
                "input_folder": {"type": "string"},
                "output_folder": {"type": "string"},
                "processed_at": {"type": "string"},
            },
        },
    },
}

AGENT2_SYSTEM = (
    "Você é o Feature Engineering Agent (AML-FT). Sua função é analisar a camada trusted "
    "e gerar um feature_pack pronto para detecção e ML. "
    "Regras: responda EXCLUSIVAMENTE em JSON válido (sem markdown). "
    "Não invente dados. Se uma feature não puder ser criada por falta de colunas, "
    "registre em warnings e proponha fallback. "
    "Só prossiga se ready_for_features=true; caso contrário, status='fail'. "
    "Entregue dicionário de features (definição + colunas de origem) e cobertura (% preenchimento)."
)

Define, via JSON Schema, o formato exato da resposta do Agent 2 e proíbe campos extras.
- Campos obrigatórios: status, transaction_features, entity_features, notes.
- Pontos-chave:
  - status: estado da especificação de features (ok | warn | fail).
  - transaction_features: array de features calculáveis a partir de dados de transação (name, definition, requires_cols que lista colunas necessárias).
  - entity_features: objeto aninhado contendo subespecificação customer com features agregadas por cliente/entidade (name, definition, requires_cols).
  - notes: justificativas, limitações e observações sobre features propostas.

- AGENT2_SYSTEM: instrução ao LLM para propor features realistas de AML-FT (velocity, income ratio, concentration, device/MCC reuse) sem inventar colunas; cada feature deve declarar exatamente which columns it requires para validação posterior.

In [7]:
def agent2_feature_spec(trusted_tables: dict[str, pd.DataFrame], ingestion_result: dict, model: str = "gpt-4.1-mini") -> dict:
    """Gera uma especificação de features de AML-FT baseada nas colunas disponíveis e saída do Agent1.

    Args:
        trusted_tables: dicionário {nome: DataFrame} com as tabelas confiáveis (saída de Agent1).
        ingestion_result: dicionário com resultado do Agent1 (contém ready_for_features, quality_score, etc.).
        model: identificador do modelo LLM a ser usado.

    Returns:
        dict: especificação conforme `AGENT2_SCHEMA` expandido (contendo `status`, `quality_score`,
        `ready_for_ml`, `feature_pack`, etc.).

    Efeitos colaterais:
        - Persiste a especificação em `artifacts/agent2_feature_spec.json` via `save_json`.
        - Não modifica as tabelas confiáveis.
    """
    # Provide only schemas/columns to LLM
    schemas = {name: list(df.columns) for name, df in trusted_tables.items()}

    user =  f"""
    Você recebeu a saída do Agente 1 (Data Ingestion & Quality) como "ingestion_result". Use isso como gate de entrada.

    ingestion_result:
    {json.dumps(ingestion_result, ensure_ascii=False)}

    IMPORTANTE:
    - Se ingestion_result.ready_for_features != true, retorne status="fail" e explique.

    Objetivo:
    Ler as tabelas da pasta "trusted" e criar features AML-FT por transação e por entidade (cliente/merchant/device/ip, se existirem).

    Colunas disponíveis na camada trusted:
    {json.dumps(schemas, ensure_ascii=False)}

    Tarefas:
    1) Gate:
    - Se ingestion_result.ready_for_features != true, retorne status="fail" e explique.

    2) Enriquecimentos/derivações (crie somente se houver colunas):
    - country_from_ip, country_risk_bucket (se houver IP/pais)
    - mcc_risk_bucket (se houver MCC)
    - pep_normalized (se houver PEP/KYC)
    - device_reuse_7d, ip_reuse_7d (se houver device/ip)
    - cross_border_flag (se houver país origem/destino ou ip_country vs country)

    3) Features por transação:
    - amount_brl
    - rail/channel/status
    - income_ratio = amount_brl / monthly_income (se houver renda)
    - velocity_count_30m (tx count por customer em 30 min)
    - structuring_7d_count (tx count em faixa 9.5k–10k em 7d, se BRL)
    - distinct_counterparties_7d (se houver receiver/merchant/counterparty)
    - ecommerce_no_3ds_flag (se houver 3DS/ECI)

    4) Features agregadas por entidade:
    - customer: tx_count_7d, tx_sum_7d, distinct_merchants_7d, distinct_devices_7d
    - merchant: tx_count_7d, distinct_customers_7d, chargeback_rate (se houver)
    - device/ip: distinct_customers_7d

    5) Qualidade e gate trusted→features:
    - feature_coverage: % preenchimento por feature
    - status ok|warn|fail
    - quality_score 0–100
    - ready_for_ml true/false (precisa de cobertura mínima)

    Output:
    Retorne EXATAMENTE o JSON do schema.
    - feature_dataset_ref deve ser "features/feature_table.parquet"
    - provenance.processed_at use "{utc_now_iso()}"
"""
    spec = llm_json("gpt-4.1-mini", AGENT2_SYSTEM, user, "agent2_feature_spec", AGENT2_SCHEMA)
    save_json(os.path.join(ARTIFACTS_DIR, "agent2_feature_spec.json"), spec)
    return spec

def compute_features(trusted_tables: dict[str, pd.DataFrame], spec: dict) -> pd.DataFrame:
    # Expect a transactions-like table
    tx_name = "transactions" if "transactions" in trusted_tables else list(trusted_tables.keys())[0]
    tx = trusted_tables[tx_name].copy()

    # Minimal: ensure timestamp & customer_id exist for time features
    # (If your dataset uses sender_id instead of customer_id, adjust here.)
    if "customer_id" not in tx.columns and "sender_id" in tx.columns:
        tx["customer_id"] = tx["sender_id"]

    # Ensure a timestamp column
    ts_col = "timestamp" if "timestamp" in tx.columns else ("transaction_datetime" if "transaction_datetime" in tx.columns else None)
    if ts_col is None:
        raise ValueError("No timestamp column found for velocity features.")

    tx = tx.sort_values(ts_col)

    # Example deterministic features (you can expand based on spec)
    if "amount_brl" in tx.columns:
        tx["f_log_amount"] = (tx["amount_brl"].clip(lower=0.01)).apply(lambda x: float(np.log(x)))  # simplest
    else:
        tx["f_log_amount"] = 0.0

    # velocity_30m: count per customer in last 30 minutes (simple rolling)
    tx[ts_col] = pd.to_datetime(tx[ts_col], errors="coerce", utc=True)
    tx["f_velocity_30m"] = (
        tx.set_index(ts_col)
          .groupby("customer_id")["customer_id"]
          .rolling("30min").count()
          .reset_index(level=0, drop=True)
          .values
    )

    # income_ratio if available via kyc
    if "kyc_profiles" in trusted_tables:
        kyc = trusted_tables["kyc_profiles"].copy()
        if "customer_id" not in kyc.columns and "sender_id" in kyc.columns:
            kyc["customer_id"] = kyc["sender_id"]
        income_col = "monthly_income" if "monthly_income" in kyc.columns else None
        if income_col:
            tx = tx.merge(kyc[["customer_id", income_col]], on="customer_id", how="left")
            if "amount_brl" in tx.columns:
                tx["f_income_ratio"] = tx["amount_brl"] / tx[income_col]
            else:
                tx["f_income_ratio"] = pd.NA
        else:
            tx["f_income_ratio"] = pd.NA
    else:
        tx["f_income_ratio"] = pd.NA

    # Save features table
    feat_path = os.path.join(FEATURES_DIR, "feature_table.parquet")
    tx.to_parquet(feat_path, index=False)
    return tx

### **ML Scoring (Fila priorizada)**

Usa Isolation Forest para detectar anomalias e classificar transações por pontuação de risco

- Gate:
     - só prossegue se feature_engineering_result.ready_for_ml == true
- Entrada:
     - feature_engineering_result (JSON Agent 2)
     - features/feature_table.parquet (ou DF em memória)
- Saída lógica (JSON):
     - ml_prioritization_result (schema v1: ranking_scope, target_definition,
       validation.operational_threshold, ranked_list com explain_short/evidence_refs)
- Saída física:
     - models/model.pkl
     - models/scored.parquet
     - artifacts/agent3_ranking.json
- Evento emitido:
     - ranking.completed(status, batch_id)

In [8]:
AGENT3_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "schema_version", "agent", "status", "quality_score",
        "ranking_scope", "target_definition", "model", "validation",
        "ranked_list", "artifacts", "blocking_issues", "warnings",
        "assumptions", "provenance"
    ],
    "properties": {
        "schema_version": {"type": "string", "enum": ["v1"]},
        "agent": {"type": "string", "enum": ["ml_prioritization"]},
        "status": {"type": "string", "enum": ["ok", "warn", "fail"]},
        "quality_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "ranking_scope": {"type": "string", "enum": ["transaction", "customer"]},
        "target_definition": {"type": "string"},
        "model": {
            "type": "object",
            "additionalProperties": False,
            "required": ["type", "version", "notes"],
            "properties": {
                "type": {"type": "string"},
                "version": {"type": "string"},
                "notes": {"type": "string"},
            },
        },
        "validation": {
            "type": "object",
            "additionalProperties": False,
            "required": ["strategy", "metrics", "operational_threshold"],
            "properties": {
                "strategy": {"type": "string"},
                "metrics": {"type": "object", "additionalProperties": True},
                "operational_threshold": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["rule", "expected_review_volume_per_day"],
                    "properties": {
                        "rule": {"type": "string"},
                        "expected_review_volume_per_day": {"type": "integer"},
                    },
                },
            },
        },
        "ranked_list": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "rank", "entity_type", "entity_id", "risk_score",
                    "risk_tags", "explain_short", "evidence_refs"
                ],
                "properties": {
                    "rank": {"type": "integer"},
                    "entity_type": {"type": "string", "enum": ["transaction", "customer"]},
                    "entity_id": {"type": "string"},
                    "risk_score": {"type": "number"},
                    "risk_tags": {"type": "array", "items": {"type": "string"}},
                    "explain_short": {"type": "string"},
                    "evidence_refs": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["feature_names", "time_window", "related_ids"],
                        "properties": {
                            "feature_names": {"type": "array", "items": {"type": "string"}},
                            "time_window": {"type": "string"},
                            "related_ids": {"type": "array", "items": {"type": "string"}},
                        },
                    },
                },
            },
        },
        "artifacts": {
            "type": "object",
            "additionalProperties": False,
            "required": ["model_ref", "scored_dataset_ref"],
            "properties": {
                "model_ref": {"type": "string"},
                "scored_dataset_ref": {"type": "string"},
            },
        },
        "blocking_issues": {"type": "array", "items": {"type": "string"}},
        "warnings": {"type": "array", "items": {"type": "string"}},
        "assumptions": {"type": "array", "items": {"type": "string"}},
        "provenance": {
            "type": "object",
            "additionalProperties": False,
            "required": ["input_folder", "processed_at"],
            "properties": {
                "input_folder": {"type": "string"},
                "processed_at": {"type": "string"},
            },
        },
    },
}

AGENT3_SYSTEM = (
    "Você é o ML Prioritization Agent (AML-FT). Sua função é usar features para gerar "
    "um score de risco e uma lista priorizada. "
    "Regras: responda EXCLUSIVAMENTE em JSON válido (sem markdown). "
    "Não invente métricas. Se não houver labels, use abordagem de anomalia/weak labels "
    "e declare isso explicitamente. "
    "Só prossiga se ready_for_ml=true; caso contrário, status='fail'. "
    "Produza ranking com explain_short e referências de evidência."
)

In [9]:
def agent3_ml_prioritization(
    features_df: pd.DataFrame,
    feature_engineering_result: dict | None = None,
    top_n: int = 50,
    ranking_scope: str = "transaction",
    expected_review_volume_per_day: int = 50,
) -> dict:
    """
    Mantém a execução em sklearn (IsolationForest) e entrega o JSON no schema v1 exigido.
    - Gate: se feature_engineering_result.ready_for_ml != True => fail
    - Sem labels => anomaly score (declara target_definition)
    - explain_short/evidence_refs: heurística baseada em z-score das features (top 3)
    """

    # -----------------------------
    # 1) Gate (se houver resultado do Agente 2)
    # -----------------------------
    blocking_issues: list[str] = []
    warnings: list[str] = []
    assumptions: list[str] = []

    if feature_engineering_result is not None:
        if feature_engineering_result.get("ready_for_ml") is not True:
            return {
                "schema_version": "v1",
                "agent": "ml_prioritization",
                "status": "fail",
                "quality_score": 0,
                "ranking_scope": ranking_scope,
                "target_definition": "N/A (gate failed: ready_for_ml=false)",
                "model": {"type": "", "version": "v1", "notes": "Gate failed"},
                "validation": {
                    "strategy": "none",
                    "metrics": {},
                    "operational_threshold": {"rule": "", "expected_review_volume_per_day": 0},
                },
                "ranked_list": [],
                "artifacts": {"model_ref": os.path.join(MODELS_DIR, "model.pkl"), "scored_dataset_ref": os.path.join(MODELS_DIR, "scored.parquet")},
                "blocking_issues": ["feature_engineering_result.ready_for_ml != true"],
                "warnings": [],
                "assumptions": [],
                "provenance": {"input_folder": "features", "processed_at": utc_now_iso()},
            }

    # -----------------------------
    # 2) Preparar matriz de features
    # -----------------------------
    candidate_cols = [c for c in features_df.columns if c.startswith("f_")]
    if not candidate_cols:
        return {
            "schema_version": "v1",
            "agent": "ml_prioritization",
            "status": "fail",
            "quality_score": 0,
            "ranking_scope": ranking_scope,
            "target_definition": "Anomaly detection (no labels) — but no f_* features found",
            "model": {"type": "IsolationForest", "version": "v1", "notes": "No feature columns"},
            "validation": {
                "strategy": "none",
                "metrics": {},
                "operational_threshold": {"rule": "", "expected_review_volume_per_day": 0},
            },
            "ranked_list": [],
            "artifacts": {"model_ref": os.path.join(MODELS_DIR, "model.pkl"), "scored_dataset_ref": os.path.join(MODELS_DIR, "scored.parquet")},
            "blocking_issues": ["No feature columns starting with 'f_' were found."],
            "warnings": [],
            "assumptions": [],
            "provenance": {"input_folder": "features", "processed_at": utc_now_iso()},
        }

    df = features_df.copy()

    # Para manter a estrutura, priorizamos transação; customer ranking exigiria agregação extra
    if ranking_scope == "customer":
        warnings.append("ranking_scope='customer' solicitado, mas esta implementação prioriza transações sem agregação.")
        ranking_scope = "transaction"

    # Identificador principal
    entity_id_col = "transaction_id" if "transaction_id" in df.columns else None
    if entity_id_col is None:
        # fallback: usa índice como id
        assumptions.append("transaction_id ausente; usando índice como entity_id.")
        df["_entity_id"] = df.index.astype(str)
        entity_id_col = "_entity_id"

    X = df[candidate_cols].fillna(0.0).astype(float)

    # -----------------------------
    # 3) Treinar modelo (sem labels => anomalia)
    # -----------------------------
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xs = scaler.fit_transform(X)

    model = IsolationForest(
        n_estimators=200,
        contamination="auto",
        random_state=42,
    )
    model.fit(Xs)

    # scores: maior = mais anômalo
    scores_raw = -model.decision_function(Xs)
    df["risk_score_raw"] = scores_raw
    df["risk_score"] = df["risk_score_raw"].rank(pct=True)

    # -----------------------------
    # 4) Explain_short/evidence_refs (heurística com z-score nas features)
    # -----------------------------
    # z-score por feature para destacar quais variáveis estão mais extremas no item
    # (não é "feature contribution" real do IsolationForest; declaramos como heurística)
    mu = X.mean(axis=0).to_numpy()
    sd = X.std(axis=0, ddof=0).replace(0, 1e-9).to_numpy()
    Z = (X.to_numpy() - mu) / sd
    absZ = np.abs(Z)

    ranked = df.sort_values("risk_score", ascending=False).head(top_n).copy()

    ranked_list = []
    for i, row in enumerate(ranked.itertuples(index=False), start=1):
        row_dict = row._asdict()

        # achar índice do row no df original para puxar absZ
        # (mais simples: localizar por risk_score_raw + entity_id; em datasets grandes isso pode ser pesado,
        # mas mantém estrutura e evita depender de índices)
        ent_id = str(row_dict.get(entity_id_col, ""))
        # tentar localizar o primeiro match por entity_id
        idx_candidates = df.index[df[entity_id_col].astype(str) == ent_id].tolist()
        z_idx = idx_candidates[0] if idx_candidates else None

        top_features = []
        if z_idx is not None:
            j = df.index.get_loc(z_idx) if hasattr(df.index, "get_loc") else None
            if j is not None:
                top_idx = np.argsort(-absZ[j])[:3]
                top_features = [candidate_cols[k] for k in top_idx]
        if not top_features:
            top_features = candidate_cols[:3]

        # tags heurísticas baseadas em nomes de features
        risk_tags = []
        if any("velocity" in f for f in top_features):
            risk_tags.append("velocity")
        if any("income" in f or "ratio" in f for f in top_features):
            risk_tags.append("income_mismatch")
        if any("struct" in f for f in top_features):
            risk_tags.append("structuring")
        if not risk_tags:
            risk_tags = ["anomaly"]

        explain_short = (
            f"Anomalia (sem labels) priorizada por IsolationForest. "
            f"Principais drivers heurísticos (z-score): {', '.join(top_features)}."
        )

        ranked_list.append({
            "rank": i,
            "entity_type": "transaction",
            "entity_id": ent_id,
            "risk_score": float(row_dict.get("risk_score", 0.0)),
            "risk_tags": risk_tags[:5],
            "explain_short": explain_short,
            "evidence_refs": {
                "feature_names": top_features,
                "time_window": "N/A",
                "related_ids": (
                    [str(row_dict["customer_id"])]
                    if "customer_id" in row_dict and pd.notna(row_dict["customer_id"])
                    else []
                ),
            }
        })

    # -----------------------------
    # 5) Validação/threshold operacional (sem labels)
    # -----------------------------
    # Corte operacional: top 1% como default (pode ajustar)
    operational_rule = "top_1_percent"
    metrics = {
        "score_distribution": {
            "min": float(df["risk_score"].min()),
            "p50": float(df["risk_score"].quantile(0.50)),
            "p95": float(df["risk_score"].quantile(0.95)),
            "max": float(df["risk_score"].max()),
        },
        "note": "Sem labels; métricas supervisionadas não aplicáveis. Distribuição de score reportada."
    }

    # -----------------------------
    # 6) Artefatos (model + scored)
    # -----------------------------
    model_ref = os.path.join(MODELS_DIR, "model.pkl")
    scored_ref = os.path.join(MODELS_DIR, "scored.parquet")

    # dataset scored completo (ou pelo menos com colunas essenciais)
    scored_cols = [entity_id_col, "risk_score", "risk_score_raw"] + candidate_cols
    keep_cols = [c for c in scored_cols if c in df.columns]
    df[keep_cols].to_parquet(scored_ref, index=False)

    # salvar modelo + scaler (para reproduzir)
    with open(model_ref, "wb") as f:
        pickle.dump({"scaler": scaler, "model": model, "feature_cols": candidate_cols}, f)

    # -----------------------------
    # 7) Montar output no schema v1
    # -----------------------------
    # quality_score simples: baseado em % de preenchimento médio nas f_*
    fill_rate = 1.0 - float(X.isna().mean().mean())
    quality_score = int(round(100 * max(0.0, min(1.0, fill_rate))))

    status = "ok" if quality_score >= 70 else ("warn" if quality_score >= 40 else "fail")
    if status == "fail":
        blocking_issues.append("Low feature fill rate; insufficient quality for prioritization.")
    if status == "warn":
        warnings.append("Feature fill rate moderate; review coverage and imputation strategy.")

    out = {
        "schema_version": "v1",
        "agent": "ml_prioritization",
        "status": status,
        "quality_score": quality_score,
        "ranking_scope": "transaction",
        "target_definition": "Unsupervised anomaly score (no labels). Higher risk_score => more anomalous behavior.",
        "model": {"type": "IsolationForest", "version": "v1", "notes": "StandardScaler + IsolationForest; explain_short uses z-score heuristics."},
        "validation": {
            "strategy": "no_labels_anomaly_scoring",
            "metrics": metrics,
            "operational_threshold": {
                "rule": operational_rule,
                "expected_review_volume_per_day": int(expected_review_volume_per_day),
            },
        },
        "ranked_list": ranked_list,
        "artifacts": {
            "model_ref": model_ref,
            "scored_dataset_ref": scored_ref,
        },
        "blocking_issues": blocking_issues,
        "warnings": warnings,
        "assumptions": assumptions,
        "provenance": {
            "input_folder": "features",
            "processed_at": utc_now_iso(),
        },
    }

    # salva artefato JSON (mantendo padrão do seu pipeline)
    save_json(os.path.join(ARTIFACTS_DIR, "agent3_ranking.json"), out)
    return out

### **Agent 4: SAR Report Generation**

Gera um Relatório de Atividade Suspeita a partir de transações de alto risco usando LLM.

- Entrada:
     - ml_prioritization_result (JSON Agent 3)
     - features/feature_table.parquet
     - models/scored.parquet
     - trusted/transactions.parquet (e outros se necessário)
- Saída lógica (JSON):
     - sar_report_result (schema v1: selected_case, sar_structured, sar_markdown,
       evidence_refs, warnings)
- Saída física:
     - artifacts/agent4_sar.json
- Evento emitido:
     - sar.completed(status, case_id, batch_id)

In [10]:
AGENT4_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "schema_version", "agent", "status", "quality_score",
        "selected_case", "sar_structured", "sar_markdown",
        "evidence_refs", "blocking_issues", "warnings",
        "assumptions", "provenance"
    ],
    "properties": {
        "schema_version": {"type": "string", "enum": ["v1"]},
        "agent": {"type": "string", "enum": ["sar_report"]},
        "status": {"type": "string", "enum": ["ok", "warn", "fail"]},
        "quality_score": {"type": "integer", "minimum": 0, "maximum": 100},
        "selected_case": {
            "type": "object",
            "additionalProperties": False,
            "required": ["entity_type", "entity_id", "related_transactions", "time_window"],
            "properties": {
                "entity_type": {"type": "string", "enum": ["transaction", "customer"]},
                "entity_id": {"type": "string"},
                "related_transactions": {"type": "array", "items": {"type": "string"}},
                "time_window": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["start", "end"],
                    "properties": {"start": {"type": "string"}, "end": {"type": "string"}},
                },
            },
        },
        "sar_structured": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "case_identification", "executive_summary", "typologies",
                "signals_alerts_triggered", "detailed_analysis",
                "legal_basis", "recommended_actions", "annexes"
            ],
            "properties": {
                "case_identification": {"type": "object", "additionalProperties": False, "properties": {}},
                "executive_summary": {"type": "string"},
                "typologies": {"type": "array", "items": {"type": "string"}},
                "signals_alerts_triggered": {"type": "array", "items": {"type": "string"}},
                "detailed_analysis": {
                    "type": "object",
                    "additionalProperties": False,
                    "required": ["timeline", "key_tables", "network_summary"],
                    "properties": {
                        "timeline": {"type": "array", "items": {"type": "object", "additionalProperties": False, "properties": {}}},
                        "key_tables": {"type": "array", "items": {"type": "string"}},
                        "network_summary": {"type": "object", "additionalProperties": False, "properties": {}},
                    },
                },
                "legal_basis": {"type": "array", "items": {"type": "string"}},
                "recommended_actions": {"type": "array", "items": {"type": "string"}},
                "annexes": {"type": "array", "items": {"type": "string"}},
            },
        },
        "sar_markdown": {"type": "string"},
        "evidence_refs": {"type": "array", "items": {"type": "string"}},
        "blocking_issues": {"type": "array", "items": {"type": "string"}},
        "warnings": {"type": "array", "items": {"type": "string"}},
        "assumptions": {"type": "array", "items": {"type": "string"}},
        "provenance": {
            "type": "object",
            "additionalProperties": False,
            "required": ["inputs", "processed_at"],
            "properties": {
                "inputs": {"type": "array", "items": {"type": "string"}},
                "processed_at": {"type": "string"},
            },
        },
    },
}

AGENT4_SYSTEM = (
    "Você é o SAR Report Agent (AML-FT). Sua função é selecionar o caso mais crítico do ranking "
    "e produzir um SAR completo e pronto para reporte. "
    "Regras: responda EXCLUSIVAMENTE em JSON válido (sem markdown), contendo também sar_markdown. "
    "Não invente fatos: todas as afirmações devem referenciar evidências (IDs e campos). "
    "O SAR deve seguir formato fixo com seções obrigatórias. "
    "Se faltarem evidências, marque status='warn' e descreva lacunas."
)

In [11]:
def agent4_generate_sar(ranking: dict, scored_top_row: dict, model: str = "gpt-4.1-mini") -> dict:
    user = f"""
ml_prioritization_result:
{json.dumps(ranking, ensure_ascii=False)}

Você também tem acesso aos datasets referenciados:
- features/feature_table.parquet
- models/scored.parquet
- trusted/transactions.parquet (e demais se necessário)

Objetivo:
Selecionar o caso MAIS CRÍTICO e gerar um SAR completo.

Tarefas:
1) Seleção:
- Escolha rank=1 (ou justifique outro) como caso principal.
- Defina escopo: cliente e transações relacionadas.

2) Evidências:
- Construa resumo de evidências: velocity/structuring, outliers de score, contrapartes,
  device/ip/geo/mcc/kyc se existirem.
- Monte timeline com timestamps e valores (se não houver timestamps, registre limitação).

3) SAR completo com seções:
- Identificação do Caso
- Resumo Executivo
- Sinais/Alertas Acionados
- Análise Detalhada (timeline + tabelas essenciais)
- Base Legal/Regulatória (alto nível: FATF/COAF/BACEN)
- Conclusão e Ações Recomendadas
- Anexos

Output:
Retorne EXATAMENTE o JSON do schema.
- provenance.inputs deve ser ["ml_prioritization_result"]
- provenance.processed_at use "{utc_now_iso()}"
"""
    sar = llm_json(model, AGENT4_SYSTEM, user, "agent4_sar", AGENT4_SCHEMA)
    save_json(os.path.join(ARTIFACTS_DIR, "agent4_sar.json"), sar)
    return sar

### **Pipeline**


In [12]:
def run_pipeline(model: str = "gpt-4.1-mini", top_n: int = 50):
    """
    Execute the complete AML-FT pipeline:
    1. Load raw data
    2. Agent 1: quality plan + apply trusted
    3. Agent 2: feature spec + compute features
    4. Agent 3: anomaly ranking (schema v1)
    5. Agent 4: SAR for top case
    """
    raw_tables = load_raw_tables(RAW_DIR)

    # Agent 1
    plan = agent1_ingest_and_quality(raw_tables, model=model)
    if plan["status"] == "fail":
        raise RuntimeError(f"Agent1 failed: {plan.get('notes')}")

    trusted_tables = apply_agent1_plan(raw_tables, plan)

    # Agent 2
    spec = agent2_feature_spec(trusted_tables, plan, model=model)
    if spec["status"] == "fail":
        raise RuntimeError(f"Agent2 failed: {spec.get('notes')}")

    features_df = compute_features(trusted_tables, spec)

    feature_engineering_result_min = {
        "ready_for_ml": True,
        "quality_score": 70,
        "feature_dataset_ref": os.path.join(FEATURES_DIR, "feature_table.parquet"),
    }

    # Agent 3 (novo)
    ranking = agent3_ml_prioritization(
        features_df,
        feature_engineering_result=feature_engineering_result_min,
        top_n=top_n,
        ranking_scope="transaction",
        expected_review_volume_per_day=50
    )

    if ranking["status"] == "fail" or not ranking.get("ranked_list"):
        raise RuntimeError(f"Agent3 failed: {ranking.get('blocking_issues') or ranking.get('warnings')}")

    # pick top item for SAR (novo schema do Agent 3)
    top_item = ranking["ranked_list"][0]
    entity_id = top_item.get("entity_id", "")
    entity_type = top_item.get("entity_type", "transaction")

    # construir evidência mínima para o Agent 4 sem inventar campos
    scored_top_row = {
        "entity_type": entity_type,
        "entity_id": entity_id,
        "risk_score": top_item.get("risk_score", 0.0),
        "risk_tags": top_item.get("risk_tags", []),
        "explain_short": top_item.get("explain_short", ""),
        "evidence_refs": top_item.get("evidence_refs", {}),
    }

    # Agent 4
    sar = agent4_generate_sar(ranking, scored_top_row, model=model)
    return {"agent1": plan, "agent2": spec, "agent3": ranking, "agent4": sar}

In [ ]:
if __name__ == "__main__":
    results = run_pipeline(model="gpt-4.1-mini", top_n=50)
    print("Pipeline completed successfully!")
    print("SAR Report Status:", results["agent4"]["status"])

Pipeline completed successfully!
SAR Report Status: ok
